# Why Clean and Prepare Data

Real-world data is rarely ready for analysis when it is first collected.  
It often contains missing values, inconsistencies, incorrect types, and errors that can lead to misleading results.

Cleaning and preparing data ensures that:
- analyses are accurate and reliable
- calculations behave as expected
- results can be trusted and reproduced
- downstream tasks such as visualization, modelling, and reporting are simpler

In practice, data preparation is one of the most important and time-consuming steps in any data analysis workflow.


## Note: There is no single “correct” way to handle missing values.
The appropriate strategy depends on the data, context, and downstream analysis.

# Import packages

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=FutureWarning) 

import pandas as pd
import numpy as np

# Handling Missing Values

Missing values are common in real-world datasets and can occur for many reasons, such as data collection issues or incomplete records.  
If not handled carefully, missing values can break calculations or bias analysis results.

This step focuses on deciding whether to:
- fill missing values using sensible defaults or estimates
- remove records that cannot be reliably used

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "age": [25, 30, np.nan, 40],
    "salary": [50000, np.nan, 60000, 70000],
    "department": ["IT", "HR", None, "IT"]
})

df

## `isna()` – Detect missing (NaN) values

In [ ]:
df.isna()

## `notna()` – Identify non-null entries

In [ ]:
df.notna()

## `dropna()` – Remove rows or columns

In [ ]:
df.dropna()

In [ ]:
df.dropna(axis=1)

## `fillna()` – Replace missing values

In [ ]:
df.fillna(0)

In [ ]:
df["department"].fillna("Unknown")

## `interpolate()` – Fill gaps using trends

In [ ]:
df["salary"].interpolate()

## `mean()` – Numeric imputation

In [ ]:
df["salary"].fillna(df["salary"].mean())

## `median()` – Robust imputation for skewed data

In [ ]:
df["salary"].fillna(df["salary"].median())

## `mode()` – Categorical imputation

In [ ]:
df["department"].fillna(df["department"].mode()[0])

# Removing Duplicates

Duplicate records can lead to double-counting and distorted results.  
They often arise from data merges, system errors, or repeated data collection.

Removing duplicates helps ensure that each real-world entity or event is represented only once in the dataset.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "policy_id": [101, 101, 102, 103, 103],
    "customer": ["A", "A", "B", "C", "C"],
    "premium": [500, 500, 600, 700, 700]
})

## `duplicated()` – Detect duplicate rows

In [ ]:
df.duplicated()

## `drop_duplicates()` – Remove duplicate rows

In [ ]:
df.drop_duplicates()

In [ ]:
df.duplicated(subset=["policy_id"])

In [ ]:
df.drop_duplicates(subset=["policy_id"], keep="first")

In [ ]:
df.drop_duplicates(subset=["policy_id"], keep="last")

## `value_counts()` – Identify repeated values

In [ ]:
df["policy_id"].value_counts()

## `groupby()` – Aggregate data by duplicate keys

In [ ]:
df.groupby("policy_id").agg({
    "premium": "sum"
})

## `reset_index()` – Rebuild index after cleaning

In [ ]:
df_clean = df.drop_duplicates(subset=["policy_id"])
df_clean.reset_index(drop=True)

# Fixing Data Types

Data may be stored using incorrect types, such as numbers stored as text or dates stored as strings.  
This can prevent calculations, comparisons, and sorting from working correctly.

Fixing data types ensures that each column behaves as expected during analysis.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "policy_id": ["101", "102", "103"],
    "premium": ["500", "600", "invalid"],
    "start_date": ["2023-01-15", "2023-02-01", "2023/03/10"]
})

df

## `dtype` – Inspect column data types

In [ ]:
df.dtypes

## `astype()` – Convert data types

In [ ]:
df["policy_id"] = df["policy_id"].astype(int)
df

## `to_datetime()` – Fix and standardize date columns

In [ ]:
df["start_date"] = pd.to_datetime(df["start_date"], format="mixed")
df

## `to_numeric()` – Convert numeric text

In [ ]:
df["premium"] = pd.to_numeric(df["premium"], errors="coerce")
df

In [ ]:
# Option 1: Convert to numeric and handle errors by setting them to NaN
df["premium"] = pd.to_numeric(df["premium"], errors='coerce')

# Option 2: If you want to see which values are causing problems first
print(df[pd.to_numeric(df["premium"], errors='coerce').isna()])

# Option 3: If you need to handle the invalid values differently
df["premium"] = df["premium"].replace("invalid", np.nan)
df["premium"] = pd.to_numeric(df["premium"])

## `dt.year` – Extract year

In [ ]:
df["start_year"] = df["start_date"].dt.year
df

## `dt.month` – Extract month

In [ ]:
df["start_month"] = df["start_date"].dt.month
df

## `tz_localize()` – Assign timezone to datetime data

In [ ]:
df["start_date"] = df["start_date"].dt.tz_localize("UTC")
df

# Cleaning Text Data

Text data often contains inconsistencies such as mixed case, extra spaces, or unexpected characters.  
These issues can cause problems when grouping, filtering, or matching values.

Cleaning text data helps standardise values and improves the reliability of analysis.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "customer_name": [" Alice ", "BOB", "Charlie"],
    "country": ["uk", "U.S.A.", "United Kingdom"],
    "policy_code": ["POL-123", "POL-456", "POL-789"],
    "notes": ["Renewal due", "High risk customer", "Premium customer"]
})

df

## `str.lower()` – Normalize text to lowercase

In [ ]:
df["customer_name"] = df["customer_name"].str.lower()
df

## `str.upper()` – Enforce consistent uppercase text

In [ ]:
df["country"] = df["country"].str.upper()
df

## `str.strip()` – Remove leading and trailing spaces

In [ ]:
df["customer_name"] = df["customer_name"].str.strip()
df

## `str.replace()` – Fix inconsistent or incorrect text

In [ ]:
df["country"] = df["country"].str.replace("U.S.A.", "USA")
df

## `str.contains()` – Identify patterns or keywords

In [ ]:
df["is_high_risk"] = df["notes"].str.contains("risk", case=False)
df

## `str.split()` – Break compound strings into parts

In [ ]:
df[["policy_prefix", "policy_number"]] = df["policy_code"].str.split("-", expand=True)
df

## `str.len()` – Validate text length consistency

In [ ]:
df["policy_code_length"] = df["policy_code"].str.len()
df

## `str.extract()` – Extract patterns using regex

In [ ]:
df["policy_number"] = df["policy_code"].str.extract(r"(\d+)")
df

# Handling Outliers

Outliers are extreme values that differ significantly from the rest of the data.  
They may represent genuine events or data errors.

Handling outliers involves identifying them and deciding whether to keep, adjust, or remove them based on context.

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "claim_amount": [1000, 1200, 1100, 50000, 1300, -50]
})

df

## `describe()` – Review value distribution

In [ ]:
df["claim_amount"].describe()

## `quantile()` – Define IQR-based boundaries

In [ ]:
q1 = df["claim_amount"].quantile(0.25)
q3 = df["claim_amount"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

lower, upper

## `between()` – Filter values within valid ranges

In [ ]:
df[df["claim_amount"].between(lower, upper)]

## `clip()` – Cap extreme values

In [ ]:
df["claim_amount_capped"] = df["claim_amount"].clip(lower=lower, upper=upper)
df

## `mean()` – Reference central tendency

In [ ]:
df["claim_amount"].mean()

## `std()` – Measure data spread

In [ ]:
df["claim_amount"].std()

## `abs()` – Remove sign-based noise

In [ ]:
df["claim_amount_absolute"] = df["claim_amount"].abs()
df

## `np.where()` – Apply conditional replacements

In [ ]:
df["claim_amount_cleaned"] = np.where(
    df["claim_amount"] < 0,
    np.nan,
    df["claim_amount"]
)

df

# Standardizing Values

The same real-world value can be represented in multiple ways across a dataset.  
This makes grouping and comparison difficult.

Standardising values ensures consistent representation of categories and labels.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "country": ["UK", "U.K.", "United Kingdom", "us", "USA"],
    "premium": [1234.567, 1234.432, 1234.5, 999.999, 1000.001],
    "commission_rate": [0.1, 0.12, 0.08, 0.15, 0.2]
})

df

## `replace()` – Correct inconsistent labels

In [ ]:
df["country"] = df["country"].replace({
    "U.K.": "UK",
    "United Kingdom": "UK",
    "us": "US"
})

df


## `map()` – Apply controlled value mapping

In [ ]:
country_map = {
    "UK": "GBR",
    "US": "USA"
}

df["country_code"] = df["country"].map(country_map)
df

## `round()` – Standardize decimal precision

In [ ]:
df["premium_rounded"] = df["premium"].round(2)
df

## `apply()` – Apply custom transformation logic

In [ ]:
df["premium_band"] = df["premium"].apply(
    lambda x: "High" if x >= 1200 else "Standard"
)

df

## `astype()` – Normalize data types

In [ ]:
df["commission_rate"] = df["commission_rate"].astype(float)
df.dtypes

## `mul()` – Convert values to new units

In [ ]:
df["commission_percent"] = df["commission_rate"].mul(100)
df

## `div()` – Scale values proportionally

In [ ]:
df["commission_scaled"] = df["commission_percent"].div(100)
df

## `clip()` – Normalize values to a range

In [ ]:
df["commission_capped"] = df["commission_rate"].clip(lower=0.1, upper=0.18)
df

# Validating Data

Data should conform to expected rules, ranges, and formats.  
Invalid values can indicate data quality issues or processing errors.

Validation checks help catch problems early and improve confidence in the dataset.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "policy_id": [101, 102, 103, 103],
    "status": ["Active", "Active", "Closed", "Active"],
    "premium": [500, 600, 700, None],
    "policy_start": pd.to_datetime(
        ["2023-01-01", "2023-02-01", "2023-03-01", "2023-04-01"]
    )
})

df

## `unique()` – Check allowed values

In [ ]:
df["status"].unique()

## `nunique()` – Validate value cardinality

In [ ]:
df["policy_id"].nunique()

## `isin()` – Enforce domain constraints

In [ ]:
allowed_status = ["Active", "Closed"]
df[df["status"].isin(allowed_status)]

## `between()` – Validate numeric ranges

In [ ]:
df[df["premium"].between(0, 1000)]

## `duplicated()` – Verify key uniqueness

In [ ]:
df["policy_id"].duplicated()

## `isna().sum()` – Review missing value distribution

In [ ]:
df.isna().sum()

## `is_monotonic_increasing` – Check sequence order

In [ ]:
df["policy_start"].is_monotonic_increasing

## `assert` – Enforce data rules

In [ ]:
assert df["premium"].min() >= 0, "Premium contains negative values"

# Fixing Inconsistent Data

Inconsistent data occurs when related values contradict each other across rows or columns.  
This often happens when combining data from multiple sources.

Fixing inconsistencies ensures the dataset reflects a coherent and accurate view of the underlying data.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "policy_id": [101, 102, 103, 104],
    "status": ["Active", "active", "Closed", "Active"],
    "premium": [500, -100, 700, 800],
    "country_code": ["UK", "UK", "US", None]
})

reference_df = pd.DataFrame({
    "policy_id": [104],
    "country_code": ["US"]
})

df

## `loc[]` – Apply conditional updates

In [ ]:
df.loc[df["premium"] < 0, "premium"] = None
df

## `where()` – Replace values conditionally

In [ ]:
df["premium"] = df["premium"].where(df["premium"] >= 0)
df

## `mask()` – Inverse conditional replacement

In [ ]:
df["status"] = df["status"].mask(df["status"] == "active", "Active")
df

## `merge()` – Correct data using references

In [ ]:
df = df.merge(
    reference_df,
    on="policy_id",
    how="left",
    suffixes=("", "_ref")
)

df["country_code"] = df["country_code"].combine_first(df["country_code_ref"])
df = df.drop(columns=["country_code_ref"])
df

## `update()` – Overwrite incorrect values

In [ ]:
df.update(
    pd.DataFrame(
        {"country_code": ["US"]},
        index=df[df["policy_id"] == 103].index
    )
)

df

## `combine_first()` – Fill from trusted source

In [ ]:
fallback = pd.Series(
    ["Unknown"] * len(df),
    index=df.index
)

df["country_code"] = df["country_code"].combine_first(fallback)
df

## `drop()` – Remove invalid records

In [ ]:
df = df.drop(df[df["premium"].isna()].index)
df

## `query()` – Filter data using conditions

In [ ]:
df.query("status == 'Active' and premium >= 500")